following along the pytorch quickstart tutorial

In [1]:
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download= True,
    transform= ToTensor()
)

test_data = datasets.FashionMNIST(
    root= "data",
    train= False,
    download= True,
    transform=ToTensor()
)



100%|██████████| 26.4M/26.4M [00:00<00:00, 30.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 1.54MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 19.1MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 24.7MB/s]


In [3]:
batch_size = 64

train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data,batch_size=batch_size)

for X,y in test_dataloader:
    print(f"Shape of X [N,C,H,W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N,C,H,W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [6]:
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.SGD(model.parameters(), lr = 1e-3)


In [7]:
def train(dataloader, model, loss_fn, optimiser):
    size = len(dataloader.dataset)
    model.train()
    for batch,  (X,y) in enumerate(dataloader):

        pred = model(X)
        loss = loss_fn(pred,y)

        loss.backward()
        optimiser.step()
        optimiser.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch+1)*len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [8]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss,  correct = 0,0
    with torch.no_grad():
        for X,y in dataloader:
            X,y = X.to(device), y.to(device)
            pred =  model(X)
            test_loss += loss_fn(pred,y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [9]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimiser)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.306904  [   64/60000]
loss: 2.301695  [ 6464/60000]
loss: 2.281349  [12864/60000]
loss: 2.286711  [19264/60000]
loss: 2.266867  [25664/60000]
loss: 2.228155  [32064/60000]
loss: 2.242372  [38464/60000]
loss: 2.207899  [44864/60000]
loss: 2.206628  [51264/60000]
loss: 2.175756  [57664/60000]
Test Error: 
 Accuracy: 34.2%, Avg loss: 2.176633 

Epoch 2
-------------------------------
loss: 2.186351  [   64/60000]
loss: 2.177907  [ 6464/60000]
loss: 2.126916  [12864/60000]
loss: 2.150072  [19264/60000]
loss: 2.103259  [25664/60000]
loss: 2.036906  [32064/60000]
loss: 2.070549  [38464/60000]
loss: 1.997318  [44864/60000]
loss: 2.001962  [51264/60000]
loss: 1.932724  [57664/60000]
Test Error: 
 Accuracy: 56.1%, Avg loss: 1.934058 

Epoch 3
-------------------------------
loss: 1.966970  [   64/60000]
loss: 1.933825  [ 6464/60000]
loss: 1.829197  [12864/60000]
loss: 1.872570  [19264/60000]
loss: 1.764733  [25664/60000]
loss: 1.707587  [32064/600